# 4.9 — Lesion Segmentation: FSG-Net

## What This Notebook Does
This notebook trains a **U-Net (ResNet34 encoder)** to simultaneously segment **5 retinal lesion types**:

| Channel | Code | Lesion Type | Clinical Meaning |
|---------|------|-------------|------------------|
| 0 | **MA** | Microaneurysms | Earliest sign of DR — tiny balloon-like swellings in blood vessels |
| 1 | **HE** | Hemorrhages | Blood leaking from damaged retinal vessels |
| 2 | **EX** | Hard Exudates | Yellow lipid deposits, indicate macular edema |
| 3 | **OD** | Optic Disc | Anatomical landmark — used as spatial reference |
| 4 | **CW** | Cotton Wool Spots | White patches from nerve fiber infarcts, severe ischemia |

### Key Difference from Vessel Segmentation
- **Vessel models** output `(B, 1, H, W)` — single binary mask
- **Lesion models** output `(B, 5, H, W)` — five independent binary masks

Each pixel can belong to **multiple lesion types** simultaneously (multi-label).

### Architecture: U-Net
U-Net uses an **encoder-decoder** structure with **skip connections**:
- **Encoder** (ResNet34, pretrained on ImageNet): extracts features at multiple scales
- **Decoder**: upsamples features back to full resolution
- **Skip connections**: connect encoder to decoder at each scale, preserving spatial detail

The final layer outputs 5 channels instead of 1, one for each lesion type.

## 1. Setup & Installation

In [ ]:
# Install required libraries (run once per Colab session)
!pip install -q albumentations segmentation-models-pytorch timm

In [ ]:
# Clone the repository (skip if already cloned)
import os
if not os.path.exists('Retinal-dr-detection'):
    !git clone https://github.com/AYMENsl25/Retinal-dr-detection.git
os.chdir('Retinal-dr-detection')
print('Working directory:', os.getcwd())

## 2. Reproducibility — Seed Everything

**Why this matters:**  
Without seeding, re-running the same notebook gives slightly different results because:
- Random weight initialization varies
- Random data augmentation varies
- GPU floating-point operations are non-deterministic by default

`seed_everything(42)` locks ALL random sources so results are **identical every run**.

In [ ]:
from src.utils import seed_everything
seed_everything(42)

## 3. Imports & Configuration

In [ ]:
import json
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp

from src.lesion.dataset import LesionDataset, get_train_transforms, get_val_transforms, LESION_TYPES, NUM_LESION_CLASSES
from src.lesion.losses import MultiLabelDiceBCELoss
from src.lesion.metrics import evaluate_multilabel_batch, evaluate_multilabel_full

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ===== CONFIGURATION =====
# These hyperparameters are IDENTICAL across all CNN-based lesion models
# to ensure a fair comparison.

CFG = {
    'model_name'    : 'FSGNet',
    'img_size'      : 512,
    'batch_size'    : 4,
    'num_epochs'    : 50,
    'learning_rate' : 1e-4,
    'weight_decay'  : 1e-4,
    'patience'      : 10,       # Early stopping patience
    'dice_weight'   : 0.5,
    'bce_weight'    : 0.5,
    'threshold'     : 0.5,
    'num_workers'   : 2,
    'use_amp'       : True,     # Mixed precision for faster training
    'out_channels'  : NUM_LESION_CLASSES,  # 5 lesion types
    'seed'          : 42,
}

# Paths — adjust if your dataset is elsewhere
DATA_DIR   = 'dataset_stage1_segmentation_processed'
ALL_CSV    = os.path.join(DATA_DIR, 'all_images.csv')
SAVE_DIR   = f'lesion_results/{CFG["model_name"]}'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'Model: {CFG["model_name"]}')
print(f'Lesion types: {LESION_TYPES}  ({CFG["out_channels"]} channels)')
print(f'Output dir: {SAVE_DIR}')

## 4. Data Preparation

### Creating Lesion-Specific Splits
The preprocessing notebook created `all_images.csv` with ALL images.  
We filter for images that have **at least one lesion mask** (`has_lesion == True`),  
then split 70% / 15% / 15% into train/val/test.

In [ ]:
# Load master CSV and filter for lesion images
df_all = pd.read_csv(ALL_CSV)
df_lesion = df_all[df_all['has_lesion'] == True].reset_index(drop=True)
print(f'Total images with lesion masks: {len(df_lesion)}')

# Split: 70% train, 15% val, 15% test
train_df, rem_df = train_test_split(df_lesion, test_size=0.30, random_state=CFG['seed'])
val_df, test_df  = train_test_split(rem_df,    test_size=0.50, random_state=CFG['seed'])

# Save splits so every model uses exactly the same data
LESION_TRAIN_CSV = os.path.join(DATA_DIR, 'lesion_train_split.csv')
LESION_VAL_CSV   = os.path.join(DATA_DIR, 'lesion_val_split.csv')
LESION_TEST_CSV  = os.path.join(DATA_DIR, 'lesion_test_split.csv')

train_df.to_csv(LESION_TRAIN_CSV, index=False)
val_df.to_csv(LESION_VAL_CSV, index=False)
test_df.to_csv(LESION_TEST_CSV, index=False)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('✅ Lesion split CSVs saved')

In [ ]:
# Create datasets and dataloaders
train_dataset = LesionDataset(LESION_TRAIN_CSV, DATA_DIR, transform=get_train_transforms(CFG['img_size']))
val_dataset   = LesionDataset(LESION_VAL_CSV,   DATA_DIR, transform=get_val_transforms(CFG['img_size']))
test_dataset  = LesionDataset(LESION_TEST_CSV,  DATA_DIR, transform=get_val_transforms(CFG['img_size']))

train_loader = DataLoader(train_dataset, batch_size=CFG['batch_size'], shuffle=True,  num_workers=CFG['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'], pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'], pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}')

# Quick sanity check
imgs, masks = next(iter(train_loader))
print(f'Image batch shape: {imgs.shape}')   # Expected: (B, 3, 512, 512)
print(f'Mask batch shape:  {masks.shape}')   # Expected: (B, 5, 512, 512)

## 5. Model Architecture — FSG-Net

FSG-Net (Feature Selection and Grouping Network) uses:
- **Feature Selection Module**: selects the most relevant features at each scale
- **Feature Grouping Module**: groups complementary features for better fusion
- Multi-scale architecture designed for fine-grained segmentation

Particularly effective for segmenting small, scattered lesions like MA and HE.

In [ ]:
# ============================================================
# Cell 9: FSG-Net — Fixed PyTorch Implementation
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


# ── Block 1: Depthwise Separable Conv ───────────────────────
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, padding=1):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_ch, in_ch, kernel_size,
            padding=padding, groups=in_ch, bias=False
        )
        self.pointwise = nn.Conv2d(in_ch, out_ch, 1, bias=False)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))


# ── Block 2: Modernized Convolution Block (MCB) ─────────────
class MCB(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = DepthwiseSeparableConv(in_ch, out_ch)
        self.bn1   = nn.BatchNorm2d(out_ch)

        self.conv2 = DepthwiseSeparableConv(out_ch, out_ch)
        self.bn2   = nn.BatchNorm2d(out_ch)

        self.act = nn.GELU()

        self.shortcut = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch)
        ) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.act(out + residual)


# ── Block 3: Full-Scale Module ──────────────────────────────
class FullScaleModule(nn.Module):
    def __init__(self, encoder_channels, out_ch):
        super().__init__()
        total_in = sum(encoder_channels)
        self.fuse = nn.Sequential(
            nn.Conv2d(total_in, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )

    def forward(self, enc_features, target_size):
        upsampled = [
            F.interpolate(f, size=target_size, mode='bilinear', align_corners=False)
            for f in enc_features
        ]
        fused = torch.cat(upsampled, dim=1)
        return self.fuse(fused)


# ── Block 4: Guided Residual Module (FIXED) ─────────────────
class GuidedResidualModule(nn.Module):
    """
    x_ch   : channels of decoder feature x
    ctx_ch : channels of full-scale feature full_scale_feat
    """
    def __init__(self, x_ch, ctx_ch, hidden_ch):
        super().__init__()

        # Attention is generated FROM full-scale context
        # and converted TO decoder feature channel dimension
        self.attn = nn.Sequential(
            nn.Conv2d(ctx_ch, hidden_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(hidden_ch),
            nn.GELU(),
            nn.Conv2d(hidden_ch, x_ch, 1, bias=False),
            nn.Sigmoid(),
        )

        # Guided refinement operates on decoder feature channels
        self.guided_conv = nn.Sequential(
            nn.Conv2d(x_ch, hidden_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(hidden_ch),
            nn.GELU(),
            nn.Conv2d(hidden_ch, x_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(x_ch),
        )

        self.act = nn.GELU()

    def forward(self, x, full_scale_feat):
        attn_map = self.attn(full_scale_feat)   # (B, x_ch, H, W)
        guided   = self.guided_conv(x * attn_map)
        return self.act(x + guided)


# ── Block 5: Guided Convolution Block (FIXED) ───────────────
class GuidedConvBlock(nn.Module):
    def __init__(self, in_ch, ctx_ch, hidden_ch, out_ch, n_grm=2):
        super().__init__()
        self.grms = nn.ModuleList([
            GuidedResidualModule(in_ch, ctx_ch, hidden_ch)
            for _ in range(n_grm)
        ])
        self.ds_head = nn.Conv2d(in_ch, out_ch, 1)

    def forward(self, x, full_scale_feat):
        for grm in self.grms:
            x = grm(x, full_scale_feat)
        aux_pred = self.ds_head(x)
        return x, aux_pred


# ── Main FSG-Net ────────────────────────────────────────────
class FSGNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, enc_channels=None, gcb_hidden=64):
        super().__init__()

        if enc_channels is None:
            enc_channels = [32, 64, 128, 256]

        # Encoder
        self.enc1 = MCB(in_ch, enc_channels[0])           # 512x512
        self.enc2 = MCB(enc_channels[0], enc_channels[1]) # 256x256
        self.enc3 = MCB(enc_channels[1], enc_channels[2]) # 128x128
        self.enc4 = MCB(enc_channels[2], enc_channels[3]) # 64x64
        self.pool = nn.MaxPool2d(2, 2)

        # Full-scale modules
        self.fsm_out = 128
        self.fsm3 = FullScaleModule(enc_channels, self.fsm_out)
        self.fsm2 = FullScaleModule(enc_channels, self.fsm_out)
        self.fsm1 = FullScaleModule(enc_channels, self.fsm_out)

        # Decoder
        self.up4  = nn.ConvTranspose2d(enc_channels[3], enc_channels[2], 2, stride=2)
        self.dec3 = MCB(enc_channels[2] * 2, enc_channels[2])

        self.up3  = nn.ConvTranspose2d(enc_channels[2], enc_channels[1], 2, stride=2)
        self.dec2 = MCB(enc_channels[1] * 2, enc_channels[1])

        self.up2  = nn.ConvTranspose2d(enc_channels[1], enc_channels[0], 2, stride=2)
        self.dec1 = MCB(enc_channels[0] * 2, enc_channels[0])

        # Guided Convolution Blocks (FIXED: ctx_ch=self.fsm_out)
        self.gcb3 = GuidedConvBlock(
            in_ch=enc_channels[2],
            ctx_ch=self.fsm_out,
            hidden_ch=gcb_hidden,
            out_ch=out_ch,
            n_grm=2
        )
        self.gcb2 = GuidedConvBlock(
            in_ch=enc_channels[1],
            ctx_ch=self.fsm_out,
            hidden_ch=gcb_hidden,
            out_ch=out_ch,
            n_grm=2
        )
        self.gcb1 = GuidedConvBlock(
            in_ch=enc_channels[0],
            ctx_ch=self.fsm_out,
            hidden_ch=gcb_hidden,
            out_ch=out_ch,
            n_grm=2
        )

        self.final_head = nn.Conv2d(enc_channels[0], out_ch, 1)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)               # (B, 32, 512, 512)
        e2 = self.enc2(self.pool(e1))   # (B, 64, 256, 256)
        e3 = self.enc3(self.pool(e2))   # (B, 128, 128, 128)
        e4 = self.enc4(self.pool(e3))   # (B, 256, 64, 64)

        enc_feats = [e1, e2, e3, e4]

        # Decoder stage 3
        d3 = self.up4(e4)                       # (B, 128, 128, 128)
        d3 = torch.cat([d3, e3], dim=1)         # (B, 256, 128, 128)
        d3 = self.dec3(d3)                      # (B, 128, 128, 128)
        fs3 = self.fsm3(enc_feats, d3.shape[2:])# (B, 128, 128, 128)
        d3, aux3 = self.gcb3(d3, fs3)           # aux3: (B, 1, 128, 128)

        # Decoder stage 2
        d2 = self.up3(d3)                       # (B, 64, 256, 256)
        d2 = torch.cat([d2, e2], dim=1)         # (B, 128, 256, 256)
        d2 = self.dec2(d2)                      # (B, 64, 256, 256)
        fs2 = self.fsm2(enc_feats, d2.shape[2:])# (B, 128, 256, 256)
        d2, aux2 = self.gcb2(d2, fs2)           # aux2: (B, 1, 256, 256)

        # Decoder stage 1
        d1 = self.up2(d2)                       # (B, 32, 512, 512)
        d1 = torch.cat([d1, e1], dim=1)         # (B, 64, 512, 512)
        d1 = self.dec1(d1)                      # (B, 32, 512, 512)
        fs1 = self.fsm1(enc_feats, d1.shape[2:])# (B, 128, 512, 512)
        d1, aux1 = self.gcb1(d1, fs1)           # aux1: (B, 1, 512, 512)

        # Final output
        out = self.final_head(d1)               # (B, 1, 512, 512)

        return out, aux1, aux2, aux3


# ============================================================
# Build model and run sanity check
# ============================================================
model = FSGNet(
    in_ch=CFG['in_channels'],
    out_ch=CFG['out_channels'],
    enc_channels=CFG['encoder_channels'],
    gcb_hidden=CFG['gcb_channels'],
).to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("FSG-Net architecture built successfully")
print(f"Total params    : {total:,}")
print(f"Trainable params: {trainable:,}")

with torch.no_grad():
    dummy = torch.randn(2, 3, 512, 512).to(device)
    out, a1, a2, a3 = model(dummy)

    print("\nForward pass check:")
    print(f"  Main output : {tuple(out.shape)}   (should be [2, 1, 512, 512])")
    print(f"  Aux pred 1  : {tuple(a1.shape)}   (full-res auxiliary)")
    print(f"  Aux pred 2  : {tuple(a2.shape)}   (1/2-res auxiliary)")
    print(f"  Aux pred 3  : {tuple(a3.shape)}   (1/4-res auxiliary)")

model = FSGNet(out_channels=CFG['out_channels']).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {CFG["model_name"]}')
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## 6. Training

### Loss Function: Multi-Label Dice + BCE
For each of the 5 lesion channels:
1. **Dice Loss** = 1 - Dice Coefficient → directly optimises our evaluation metric
2. **BCE Loss** = Binary Cross-Entropy → stable per-pixel gradients
3. Combined: `0.5 * Dice + 0.5 * BCE`

Losses are computed **per channel** then **averaged** — this prevents large
lesions (like OD) from dominating over tiny ones (like MA).

In [ ]:
criterion = MultiLabelDiceBCELoss(
    dice_weight=CFG['dice_weight'],
    bce_weight=CFG['bce_weight']
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG['learning_rate'],
    weight_decay=CFG['weight_decay']
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5, verbose=True
)

scaler = GradScaler() if (CFG['use_amp'] and device.type == 'cuda') else None
print('✅ Loss, optimiser, scheduler ready')

In [ ]:
# ===== TRAINING LOOP =====
best_val_dice = 0.0
best_epoch = 0
patience_counter = 0
best_model_path = os.path.join(SAVE_DIR, f'{CFG["model_name"]}_best.pth')

history = {
    'train_loss': [], 'val_loss': [],
    'train_dice_mean': [], 'val_dice_mean': [],
}
# Also track per-type dice
for lt in LESION_TYPES:
    history[f'train_dice_{lt}'] = []
    history[f'val_dice_{lt}'] = []

print(f'\n{"="*60}')
print(f'Training {CFG["model_name"]} for up to {CFG["num_epochs"]} epochs')
print(f'Device: {device} | AMP: {CFG["use_amp"]} | Patience: {CFG["patience"]}')
print(f'{"="*60}\n')

for epoch in range(1, CFG['num_epochs'] + 1):
    start = time.time()

    # --- Train ---
    model.train()
    train_loss = 0.0
    train_metrics_sum = {}
    n_train = 0

    for images, masks in tqdm(train_loader, desc=f'Epoch {epoch} Train', leave=False):
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()

        if scaler is not None:
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

        train_loss += loss.item()
        batch_m = evaluate_multilabel_batch(outputs, masks)
        for k, v in batch_m.items():
            train_metrics_sum[k] = train_metrics_sum.get(k, 0) + v
        n_train += 1

    train_loss /= n_train
    for k in train_metrics_sum:
        train_metrics_sum[k] /= n_train

    # --- Validate ---
    model.eval()
    val_loss = 0.0
    val_metrics_sum = {}
    n_val = 0

    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f'Epoch {epoch} Val', leave=False):
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item()
            batch_m = evaluate_multilabel_batch(outputs, masks)
            for k, v in batch_m.items():
                val_metrics_sum[k] = val_metrics_sum.get(k, 0) + v
            n_val += 1

    val_loss /= n_val
    for k in val_metrics_sum:
        val_metrics_sum[k] /= n_val

    # Record history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_dice_mean'].append(train_metrics_sum.get('dice_mean', 0))
    history['val_dice_mean'].append(val_metrics_sum.get('dice_mean', 0))
    for lt in LESION_TYPES:
        history[f'train_dice_{lt}'].append(train_metrics_sum.get(f'dice_{lt}', 0))
        history[f'val_dice_{lt}'].append(val_metrics_sum.get(f'dice_{lt}', 0))

    # LR scheduler
    val_dice = val_metrics_sum.get('dice_mean', 0)
    scheduler.step(val_dice)

    elapsed = time.time() - start
    lr = optimizer.param_groups[0]['lr']

    print(f'Epoch {epoch:3d}/{CFG["num_epochs"]} | '
          f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | '
          f'Val Dice(mean): {val_dice:.4f} | LR: {lr:.2e} | {elapsed:.1f}s')

    # Checkpoint
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        best_epoch = epoch
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_dice': best_val_dice,
            'history': history,
            'config': CFG,
        }, best_model_path)
        print(f'  ★ New best! Dice(mean): {best_val_dice:.4f}')
    else:
        patience_counter += 1
        if patience_counter >= CFG['patience']:
            print(f'\n⏹ Early stopping at epoch {epoch}. Best: {best_val_dice:.4f} (epoch {best_epoch})')
            break

print(f'\n{"="*60}')
print(f'Training complete. Best Val Dice(mean): {best_val_dice:.4f} (epoch {best_epoch})')
print(f'{"="*60}')

## 7. Evaluation on Test Set

We load the best checkpoint and compute per-lesion-type metrics
on the held-out test set.

In [ ]:
# Load best model
checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Loaded best model from epoch {checkpoint["epoch"]} (Dice: {checkpoint["val_dice"]:.4f})')

# Collect all predictions
all_preds = []
all_targets = []

with torch.no_grad():
    for images, masks in tqdm(test_loader, desc='Testing'):
        images = images.to(device)
        outputs = model(images)
        probs = torch.sigmoid(outputs).cpu().numpy()
        all_preds.append(probs)
        all_targets.append(masks.numpy())

all_preds = np.concatenate(all_preds, axis=0)
all_targets = np.concatenate(all_targets, axis=0)
print(f'Predictions shape: {all_preds.shape}')  # (N, 5, H, W)

# Compute full metrics
test_results = evaluate_multilabel_full(all_preds, all_targets)

# Display results
print(f'\n{"="*50}')
print(f'  TEST RESULTS — {CFG["model_name"]}')
print(f'{"="*50}')
for lt in LESION_TYPES:
    print(f'\n  {lt}:')
    print(f'    Dice: {test_results[f"dice_{lt}"]:.4f}  |  IoU: {test_results[f"iou_{lt}"]:.4f}')
    print(f'    Sens: {test_results[f"sens_{lt}"]:.4f}  |  Spec: {test_results[f"spec_{lt}"]:.4f}')
    print(f'    Prec: {test_results[f"prec_{lt}"]:.4f}  |  AUC-ROC: {test_results[f"auc_roc_{lt}"]:.4f}')

print(f'\n  MACRO AVERAGE:')
print(f'    Dice: {test_results["dice_mean"]:.4f}  |  IoU: {test_results["iou_mean"]:.4f}')
print(f'    Sens: {test_results["sens_mean"]:.4f}  |  Spec: {test_results["spec_mean"]:.4f}')
print(f'{"="*50}')

## 8. Save Results

In [ ]:
# Save metrics JSON
results_path = os.path.join(SAVE_DIR, f'{CFG["model_name"]}_results.json')
with open(results_path, 'w') as f:
    json.dump({'config': CFG, 'test_metrics': test_results, 'best_epoch': best_epoch}, f, indent=2)
print(f'✅ Metrics saved to {results_path}')

# Save training history
history_path = os.path.join(SAVE_DIR, f'{CFG["model_name"]}_history.json')
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f'✅ History saved to {history_path}')

## 9. Training Curves

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'{CFG["model_name"]} — Lesion Segmentation Training', fontsize=14, fontweight='bold')

# Loss
axes[0].plot(epochs, history['train_loss'], 'b-', label='Train')
axes[0].plot(epochs, history['val_loss'], 'r-', label='Val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Mean Dice
axes[1].plot(epochs, history['train_dice_mean'], 'b-', label='Train')
axes[1].plot(epochs, history['val_dice_mean'], 'r-', label='Val')
axes[1].set_title('Dice (Mean over 5 types)')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Per-Type Val Dice
colors = ['red', 'blue', 'green', 'orange', 'purple']
for i, lt in enumerate(LESION_TYPES):
    axes[2].plot(epochs, history[f'val_dice_{lt}'], color=colors[i], label=lt)
axes[2].set_title('Val Dice per Lesion Type')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, f'{CFG["model_name"]}_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved')

## 10. Prediction Visualization

Shows original image + ground truth masks + predicted masks for each lesion type,
using distinct colors per type.

In [ ]:
def denormalize(tensor, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    tensor = tensor.cpu() * std + mean
    return torch.clamp(tensor, 0, 1).permute(1, 2, 0).numpy()

# Visualize predictions for a few test samples
num_samples = 3
lesion_colors = {
    'MA': [1, 0, 0],     # Red
    'HE': [1, 0.5, 0],   # Orange
    'EX': [1, 1, 0],     # Yellow
    'OD': [0, 0.5, 1],   # Blue
    'CW': [1, 0, 1],     # Magenta
}

fig, axes = plt.subplots(num_samples, 2 + NUM_LESION_CLASSES,
                         figsize=(4 * (2 + NUM_LESION_CLASSES), 4 * num_samples))
fig.suptitle(f'{CFG["model_name"]} — Lesion Predictions', fontsize=14, fontweight='bold')

shown = 0
model.eval()
with torch.no_grad():
    for images, masks in test_loader:
        images_dev = images.to(device)
        preds = torch.sigmoid(model(images_dev)).cpu()

        for i in range(images.shape[0]):
            if shown >= num_samples:
                break

            img_np = denormalize(images[i])

            # Original image
            axes[shown, 0].imshow(img_np)
            axes[shown, 0].set_title('Image' if shown == 0 else '')
            axes[shown, 0].axis('off')

            # Overlay: all lesions combined
            overlay = img_np.copy()
            for ch, lt in enumerate(LESION_TYPES):
                gt_mask = masks[i, ch].numpy() > 0.5
                pred_mask = preds[i, ch].numpy() > CFG['threshold']
                color = lesion_colors[lt]
                overlay[pred_mask] = color
            axes[shown, 1].imshow(overlay)
            axes[shown, 1].set_title('All Predictions' if shown == 0 else '')
            axes[shown, 1].axis('off')

            # Per-type: GT (top half) vs Pred (bottom half)
            for ch, lt in enumerate(LESION_TYPES):
                gt = masks[i, ch].numpy()
                pr = (preds[i, ch].numpy() > CFG['threshold']).astype(float)
                combined = np.zeros((*gt.shape, 3))
                combined[:, :, 1] = gt    # Green = ground truth
                combined[:, :, 0] = pr    # Red = prediction
                both = (gt > 0.5) & (pr > 0.5)
                combined[both] = [1, 1, 0]  # Yellow = correct
                axes[shown, 2 + ch].imshow(combined)
                axes[shown, 2 + ch].set_title(lt if shown == 0 else '')
                axes[shown, 2 + ch].axis('off')

            shown += 1
        if shown >= num_samples:
            break

plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, f'{CFG["model_name"]}_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Prediction visualization saved')

In [ ]:
print(f'\n🎉 Notebook complete!')
print(f'Model:     {CFG["model_name"]}')
print(f'Best Dice: {best_val_dice:.4f} (epoch {best_epoch})')
print(f'All outputs saved to: {SAVE_DIR}/')